In [9]:
from transformers import AutoModel
from torch.nn.utils import parameters_to_vector
import torch

# Load encoder-only models (no heads)
base_encoder = AutoModel.from_pretrained("bert-base-uncased")
taskA_encoder = AutoModel.from_pretrained("model_taskA_sst2")
taskB_encoder = AutoModel.from_pretrained("model_taskB_ner")

# Convert to vectors
theta_0 = parameters_to_vector(base_encoder.parameters()).detach()
theta_A = parameters_to_vector(taskA_encoder.parameters()).detach()
theta_B = parameters_to_vector(taskB_encoder.parameters()).detach()

# Align sizes and merge deltas
min_size = min(theta_0.size(0), theta_A.size(0), theta_B.size(0))
theta_0 = theta_0[:min_size]
theta_A = theta_A[:min_size]
theta_B = theta_B[:min_size]

delta_A = theta_A - theta_0
delta_B = theta_B - theta_0
delta_merged = (delta_A + delta_B) / 2

torch.save(delta_merged, "delta_merged_encoder_only.pt")

Some weights of BertModel were not initialized from the model checkpoint at model_taskB_ner and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiTaskBERT(nn.Module):
    def __init__(self, base_model_name="bert-base-uncased", num_labels_sst2=2, num_labels_ner=9):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.classifier_sst2 = nn.Linear(self.encoder.config.hidden_size, num_labels_sst2)
        self.classifier_ner = nn.Linear(self.encoder.config.hidden_size, num_labels_ner)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, task="sst2"):
        outputs = self.encoder(input_ids=input_ids,
                               attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        
        if task == "sst2":
            cls_output = outputs.last_hidden_state[:, 0]  # [CLS] token
            return self.classifier_sst2(cls_output)
        
        elif task == "ner":
            return self.classifier_ner(outputs.last_hidden_state)
        else:
            raise ValueError("Task must be either 'sst2' or 'ner'")

In [11]:
from torch.nn.utils import vector_to_parameters

model = MultiTaskBERT()
delta_merged = torch.load("delta_merged_encoder_only.pt")
vector_to_parameters(delta_merged, model.encoder.parameters())

torch.save(model.state_dict(), "clean_multi_task_model.pt")